# 08. Data extracts for the Tableau dashboard

**Purpose.** Write the tidy CSV files that the Tableau dashboard reads, into `tableau/data/`. Every number is precomputed here, from the validated results of notebooks 02 to 07, so that Tableau only *displays* values and never recomputes a metric. Recomputing a Sharpe ratio, a beta or a cumulative return inside Tableau (with its own aggregation, filters or date ranges) is the easiest way to break the rule that every metric is defined identically in SQL, Python, Tableau and the report.

**Files written.**

| File | One row per | Used for |
|---|---|---|
| `daily_series.csv` | trading day and instrument | growth of 100, drawdown, rolling 252-day Sharpe ratio |
| `period_metrics.csv` | period, instrument and return basis | returns, volatility, Sharpe ratio with interval, maximum drawdown |
| `sharpe_difference.csv` | period and bank | H3: Sharpe(bank) − Sharpe(IHSG) with interval and p-values |
| `capm.csv` | period and bank | beta with interval, test of beta = 1, R² with interval, intercept |
| `market_implied.csv` | period and bank | actual return against the return implied by the market exposure |
| `regimes.csv` | regime | regime labels and dates, for reference lines and captions |
| `kpi.csv` | instrument | the headline tiles: full-period price and total return, Sharpe ratio, difference from the IHSG, beta, R² |

**How to run.** Run notebooks 02, 03, 05 and 07 first (their CSV files in `reports/` are read here). Then select the project `.venv` kernel, *Restart* and *Run All*. The notebook asks for the PostgreSQL user and password (for the daily series). Every check must print PASS.

**Data licence.** The daily series are derived from Yahoo Finance prices. `tableau/data/` is excluded from git, and the files should not be published (for example, to Tableau Public) without checking the data provider's terms.

In [1]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd

ROOT = Path.cwd() if (Path.cwd() / "scripts").exists() else Path.cwd().parent
sys.path.insert(0, str(ROOT / "scripts"))
try:
    import db_access
    import metrics
except ImportError as exc:
    raise ImportError(
        f"{exc}\nThis kernel is {sys.executable}. Select the project .venv kernel "
        "(top right of the notebook, Select Kernel > Python Environments > .venv), then Restart and Run All."
    ) from exc

print("project root:", ROOT)

project root: d:\Data Analyst\Portfolio


## 1. Load the database series and the validated tables

In [2]:
REPORTS = ROOT / "reports"
NEEDED = {
    "02_performance_summary.csv": "02_performance_risk", "02_sharpe_bootstrap.csv": "02_performance_risk",
    "02_rolling_sharpe_252.csv": "02_performance_risk", "03_capm_regression.csv": "03_capm",
    "03_r2_bootstrap.csv": "03_capm", "03_capm_hac.csv": "03_capm",
    "05_summary_compare.csv": "05_total_return_sensitivity", "05_sharpe_compare.csv": "05_total_return_sensitivity",
    "07_sharpe_difference.csv": "07_sharpe_difference", "07_market_implied_returns.csv": "07_sharpe_difference",
}
missing = {f: nb for f, nb in NEEDED.items() if not (REPORTS / f).exists()}
if missing:
    raise FileNotFoundError("Missing saved tables: " + ", ".join(f"{f} (run {nb} first)" for f, nb in missing.items()))

conn = db_access.connect()
try:
    prices = db_access.load_prices(conn)
    sql = db_access.load_excess_returns(conn)
finally:
    conn.close()

summary_price = pd.read_csv(REPORTS / "02_performance_summary.csv").set_index(["period", "ticker"])
sharpe_price = pd.read_csv(REPORTS / "02_sharpe_bootstrap.csv").set_index(["period", "ticker"])
rolling_saved = pd.read_csv(REPORTS / "02_rolling_sharpe_252.csv", index_col="trade_date", parse_dates=True)
capm_tbl = pd.read_csv(REPORTS / "03_capm_regression.csv").set_index(["period", "ticker"])
r2_tbl = pd.read_csv(REPORTS / "03_r2_bootstrap.csv").set_index(["period", "ticker"])
hac_tbl = pd.read_csv(REPORTS / "03_capm_hac.csv").set_index(["period", "ticker"])
summary_cmp = pd.read_csv(REPORTS / "05_summary_compare.csv", header=[0, 1], index_col=[0, 1])
sharpe_cmp = pd.read_csv(REPORTS / "05_sharpe_compare.csv").set_index(["period", "ticker"])
h3_tbl = pd.read_csv(REPORTS / "07_sharpe_difference.csv").set_index(["period", "ticker"])
implied_tbl = pd.read_csv(REPORTS / "07_market_implied_returns.csv").set_index(["period", "ticker"])
print("prices rows:", len(prices), "| SQL view rows:", len(sql))

prices rows: 5536 | SQL view rows: 5524


## 2. Settings

In [3]:
BANKS = ["BBCA", "BBRI", "BMRI"]
MARKET = "IHSG"
TICKERS = BANKS + [MARKET]
FIRST_RETURN_DAY = pd.Timestamp("2021-01-04")
PRIOR_CLOSE_DAY = pd.Timestamp("2020-12-30")
EXPECTED_DAYS = 1381
PERIOD_ORDER = {"Full": 0, "R1": 1, "R2": 2, "R3": 3, "Stub": 4}
REGIME_LABELS = {"R1": "Low-rate", "R2": "Tightening and high plateau", "R3": "Easing", "Stub": "Re-tightening (descriptive only)"}
BASIS_LABELS = {"price": "Price return", "adj": "Total return (adjusted close)"}
OUT = ROOT / "tableau" / "data"
OUT.mkdir(parents=True, exist_ok=True)

results = []


def check(name, ok, detail=""):
    results.append(bool(ok))
    print(f"{'PASS' if ok else 'FAIL'}  {name}" + (f"  [{detail}]" if detail else ""))


close = db_access.to_wide(prices, "close")[TICKERS]
ret = metrics.simple_returns(close, FIRST_RETURN_DAY)
rate = sql.drop_duplicates("trade_date").set_index("trade_date")["rate_pct"].sort_index()
excess = metrics.excess_return(ret, metrics.daily_risk_free(rate))
labels = metrics.assign_regime(ret.index)
period_days = {"Full": ret.index}
period_days.update({name: ret.index[labels == name] for name in REGIME_LABELS})
check(f"{EXPECTED_DAYS} return days, no missing values", len(ret) == EXPECTED_DAYS and not excess.isna().any().any())

PASS  1381 return days, no missing values


## 3. `regimes.csv` and `daily_series.csv`

The daily file starts at the 30 December 2020 close (growth index 100, drawdown 0), so the lines start at the buy price. The rolling Sharpe ratio is empty for the first 251 return days, by definition.

In [4]:
regimes = pd.DataFrame([{"regime": name, "regime_order": PERIOD_ORDER[name], "label": REGIME_LABELS[name],
                         "start_date": period_days[name][0].date(), "end_date": period_days[name][-1].date(),
                         "return_days": len(period_days[name])} for name in REGIME_LABELS])

path = close.loc[PRIOR_CLOSE_DAY:]
growth = path / path.iloc[0] * 100
drawdown = metrics.drawdown_series(path) * 100
rolling = metrics.rolling_sharpe(excess, window=252).reindex(path.index)
day_regime = metrics.assign_regime(path.index).fillna("Prior close")

parts = []
for t in TICKERS:
    parts.append(pd.DataFrame({"trade_date": path.index, "ticker": t, "regime": day_regime.to_numpy(),
                               "growth_index": growth[t].to_numpy(), "drawdown_pct": drawdown[t].to_numpy(),
                               "rolling_sharpe_252": rolling[t].to_numpy()}))
daily = pd.concat(parts, ignore_index=True)

check("daily file has one row per trading day and instrument", len(daily) == len(path) * len(TICKERS), f"{len(daily)} rows")
check("growth index ends at 100 x (1 + cumulative return) from notebook 02",
      all(abs(growth[t].iloc[-1] - 100 * (1 + summary_price.loc[("Full", t), "cum_return_pct"] / 100)) < 0.006 for t in TICKERS))
check("lowest drawdown equals the maximum drawdown from notebook 02",
      all(abs(drawdown[t].min() - summary_price.loc[("Full", t), "mdd_pct"]) < 0.0051 for t in TICKERS))
d_roll = (rolling.dropna()[TICKERS] - rolling_saved[TICKERS]).abs().max().max()
check("rolling Sharpe equals the saved notebook 02 table", len(rolling.dropna()) == len(rolling_saved) and d_roll < 1e-9,
      f"max abs diff {d_roll:.2e}")

PASS  daily file has one row per trading day and instrument  [5528 rows]
PASS  growth index ends at 100 x (1 + cumulative return) from notebook 02
PASS  lowest drawdown equals the maximum drawdown from notebook 02
PASS  rolling Sharpe equals the saved notebook 02 table  [max abs diff 4.44e-16]


## 4. `period_metrics.csv`

Price basis for all four instruments (notebook 02), adjusted-close basis for the banks (notebook 05). The notebook 02 summary was saved rounded to two decimals; the adjusted values are rounded the same way here so that both bases display alike.

In [5]:
rows = []
for (period, t), r in summary_price.iterrows():
    s = sharpe_price.loc[(period, t)] if (period, t) in sharpe_price.index else None
    rows.append({"period": period, "ticker": t, "basis": BASIS_LABELS["price"], "n_days": int(r["n_days"]),
                 "cum_return_pct": r["cum_return_pct"], "ann_return_pct": r["ann_return_pct"], "ann_vol_pct": r["ann_vol_pct"],
                 "sharpe": r["sharpe"], "sharpe_ci_low": np.nan if s is None else s["ci_low_b10"],
                 "sharpe_ci_high": np.nan if s is None else s["ci_high_b10"], "mdd_pct": r["mdd_pct"]})
for (period, t) in summary_cmp.index:
    s = sharpe_cmp.loc[(period, t)] if (period, t) in sharpe_cmp.index else None
    rows.append({"period": period, "ticker": t, "basis": BASIS_LABELS["adj"], "n_days": len(period_days[period]),
                 "cum_return_pct": summary_cmp.loc[(period, t), ("cum_return_pct", "adj")],
                 "ann_return_pct": summary_cmp.loc[(period, t), ("ann_return_pct", "adj")],
                 "ann_vol_pct": summary_cmp.loc[(period, t), ("ann_vol_pct", "adj")],
                 "sharpe": summary_cmp.loc[(period, t), ("sharpe", "adj")],
                 "sharpe_ci_low": np.nan if s is None else s["low_adj"], "sharpe_ci_high": np.nan if s is None else s["high_adj"],
                 "mdd_pct": summary_cmp.loc[(period, t), ("mdd_pct", "adj")]})
period_metrics = pd.DataFrame(rows)
num = ["cum_return_pct", "ann_return_pct", "ann_vol_pct", "sharpe", "sharpe_ci_low", "sharpe_ci_high", "mdd_pct"]
period_metrics[num] = period_metrics[num].astype(float).round(2)
period_metrics.insert(1, "period_order", period_metrics["period"].map(PERIOD_ORDER))
period_metrics = period_metrics.sort_values(["basis", "period_order", "ticker"])

check("period metrics: 20 price rows and 15 adjusted rows",
      (period_metrics["basis"] == BASIS_LABELS["price"]).sum() == 20 and (period_metrics["basis"] == BASIS_LABELS["adj"]).sum() == 15)
check("annualized return is empty only for the Stub",
      period_metrics.loc[period_metrics["ann_return_pct"].isna(), "period"].eq("Stub").all()
      and period_metrics.loc[period_metrics["period"] == "Stub", "ann_return_pct"].isna().all())
non_stub = period_metrics[period_metrics["period"] != "Stub"]
check("Sharpe intervals are present for every non-Stub row and contain the Sharpe ratio",
      ((non_stub["sharpe_ci_low"] <= non_stub["sharpe"]) & (non_stub["sharpe"] <= non_stub["sharpe_ci_high"])).all())

PASS  period metrics: 20 price rows and 15 adjusted rows
PASS  annualized return is empty only for the Stub
PASS  Sharpe intervals are present for every non-Stub row and contain the Sharpe ratio


## 5. `sharpe_difference.csv`, `capm.csv`, `market_implied.csv` and `kpi.csv`

Copied from notebooks 03 and 07, with readable labels. `significant_5pct` uses the Holm-adjusted p-value (within the full-period family for the full period, across all twelve tests for the regimes), which is how the report reads the test.

In [6]:
sd = h3_tbl.reset_index()
sd["p_holm"] = np.where(sd["period"] == "Full", sd["holm_full_family"], sd["holm_all_12"])
sd["significant_5pct"] = sd["p_holm"] < 0.05
sd["label"] = sd.apply(lambda r: f"{r['delta']:+.2f} (p = {r['p_value']:.2f})", axis=1)
sharpe_difference = sd[["period", "ticker", "sharpe_bank", "sharpe_ihsg", "delta", "ci_low", "ci_high", "p_value", "p_holm",
                        "significant_5pct", "label"]].copy()
sharpe_difference.insert(1, "period_order", sharpe_difference["period"].map(PERIOD_ORDER))

cp = capm_tbl.reset_index()
cp = cp.merge(r2_tbl[["ci_low_b10", "ci_high_b10"]].rename(columns={"ci_low_b10": "r2_ci_low", "ci_high_b10": "r2_ci_high"}).reset_index(),
              on=["period", "ticker"], how="left")
cp = cp.merge(hac_tbl[["p_hac_auto", "holm_hc3"]].reset_index(), on=["period", "ticker"], how="left")
capm = cp[["period", "ticker", "n", "beta", "beta_ci_low", "beta_ci_high", "p_vs_1", "holm_hc3", "p_hac_auto",
           "r2", "r2_ci_low", "r2_ci_high", "alpha_ann_pct", "alpha_p"]].rename(
    columns={"p_vs_1": "p_beta_eq_1", "holm_hc3": "p_beta_eq_1_holm", "p_hac_auto": "p_beta_eq_1_newey_west"})
capm.insert(1, "period_order", capm["period"].map(PERIOD_ORDER))

market_implied = implied_tbl.reset_index()
market_implied.insert(1, "period_order", market_implied["period"].map(PERIOD_ORDER))

pm = period_metrics.set_index(["basis", "period", "ticker"])
sdi = sharpe_difference.set_index(["period", "ticker"])
cpi = capm.set_index(["period", "ticker"])
kpi_rows = []
for t in TICKERS:
    row = {"ticker": t, "price_return_pct": pm.loc[(BASIS_LABELS["price"], "Full", t), "cum_return_pct"],
           "total_return_pct": pm.loc[(BASIS_LABELS["adj"], "Full", t), "cum_return_pct"] if t in BANKS else np.nan,
           "sharpe": pm.loc[(BASIS_LABELS["price"], "Full", t), "sharpe"]}
    if t in BANKS:
        row.update({"sharpe_diff_vs_ihsg": round(sdi.loc[("Full", t), "delta"], 2), "sharpe_diff_p": round(sdi.loc[("Full", t), "p_value"], 2),
                    "beta": round(cpi.loc[("Full", t), "beta"], 2), "r2": round(cpi.loc[("Full", t), "r2"], 2)})
    kpi_rows.append(row)
kpi = pd.DataFrame(kpi_rows)

check("KPI table: one row per instrument, full period", len(kpi) == 4 and kpi.loc[kpi["ticker"] != MARKET].notna().all().all())
check("H3 table: 12 rows, values identical to notebook 07",
      len(sharpe_difference) == 12 and np.allclose(sharpe_difference.set_index(["period", "ticker"])["delta"], h3_tbl["delta"]))
check("CAPM table: 12 rows, no missing interval or Holm value", len(capm) == 12 and not capm.isna().any().any())
check("market-implied table: 12 rows, gap = actual - implied",
      len(market_implied) == 12 and np.allclose(market_implied["gap_pct_points"], market_implied["actual_pct"] - market_implied["market_implied_pct"]))

PASS  KPI table: one row per instrument, full period
PASS  H3 table: 12 rows, values identical to notebook 07
PASS  CAPM table: 12 rows, no missing interval or Holm value
PASS  market-implied table: 12 rows, gap = actual - implied


## 6. Write the files

In [7]:
files = {"kpi.csv": kpi, "regimes.csv": regimes, "daily_series.csv": daily, "period_metrics.csv": period_metrics,
         "sharpe_difference.csv": sharpe_difference, "capm.csv": capm, "market_implied.csv": market_implied}
for name, df in files.items():
    df.to_csv(OUT / name, index=False, date_format="%Y-%m-%d")
    print(f"{name:24s} {len(df):5d} rows, {df.shape[1]} columns")

check("all seven files were written", all((OUT / n).exists() for n in files))
n_fail = results.count(False)
print(f"{len(results) - n_fail} passed, {n_fail} failed")
assert n_fail == 0, "At least one check failed. Do not use these extracts."

kpi.csv                      4 rows, 8 columns
regimes.csv                  4 rows, 6 columns


daily_series.csv          5528 rows, 6 columns
period_metrics.csv          35 rows, 12 columns
sharpe_difference.csv       12 rows, 12 columns
capm.csv                    12 rows, 15 columns
market_implied.csv          12 rows, 8 columns
PASS  all seven files were written
13 passed, 0 failed
